In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import (
    mutual_info_classif,
    chi2,
    VarianceThreshold
)
from sklearn.preprocessing import MinMaxScaler


# =========================================================
# 1. تحميل البيانات من UCI
# =========================================================

url = (
    "https://archive.ics.uci.edu/"
    "ml/machine-learning-databases/"
    "breast-cancer-wisconsin/wdbc.data"
)

# أسماء الأعمدة الموجودة في Dataset
columns = [
    "ID",
    "Diagnosis",

    "Radius_Mean",
    "Texture_Mean",
    "Perimeter_Mean",
    "Area_Mean",
    "Smoothness_Mean",
    "Compactness_Mean",
    "Concavity_Mean",
    "Concave_Points_Mean",
    "Symmetry_Mean",
    "Fractal_Dimension_Mean",

    "Radius_SE",
    "Texture_SE",
    "Perimeter_SE",
    "Area_SE",
    "Smoothness_SE",
    "Compactness_SE",
    "Concavity_SE",
    "Concave_Points_SE",
    "Symmetry_SE",
    "Fractal_Dimension_SE",

    "Radius_Worst",
    "Texture_Worst",
    "Perimeter_Worst",
    "Area_Worst",
    "Smoothness_Worst",
    "Compactness_Worst",
    "Concavity_Worst",
    "Concave_Points_Worst",
    "Symmetry_Worst",
    "Fractal_Dimension_Worst"
]

# قراءة البيانات
df = pd.read_csv(url, header=None, names=columns)

print("أبعاد البيانات:", df.shape)
print(df.head())



أبعاد البيانات: (569, 32)
         ID Diagnosis  Radius_Mean  Texture_Mean  Perimeter_Mean  Area_Mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   Smoothness_Mean  Compactness_Mean  Concavity_Mean  Concave_Points_Mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  Radius_Worst  Texture_Worst 

In [ ]:
# =========================================================
# 2. تنظيف وتجهيز البيانات
# =========================================================

# حذف الصفوف التي تحتوي على قيم مفقودة
df = df.dropna()


In [ ]:
# حذف ID لأنه مجرد رقم تعريفي وليس Feature مفيدة
df = df.drop(columns=["ID"])

In [ ]:
# تحويل Diagnosis إلى أرقام
# M = Malignant = 1
# B = Benign = 0
df["Diagnosis_Code"] = df["Diagnosis"].map({
    "M": 1,
    "B": 0
})

# التأكد من أن التحويل تم بنجاح
if df["Diagnosis_Code"].isna().any():
    raise ValueError("يوجد تصنيف غير معروف داخل عمود Diagnosis")

In [ ]:
# تحديد الـ Target
y = df["Diagnosis_Code"].astype(int)

In [ ]:

# تحديد الميزات فقط
feature_names = [column for column in columns if column not in ["ID", "Diagnosis"]]
X = df[feature_names].astype(float)

print("\nعدد الميزات:", len(feature_names))
print("عدد السجلات:", len(df))



عدد الميزات: 30
عدد السجلات: 569


# 3. first methode: Correlation

In [ ]:
# حساب الارتباط بين كل Feature والـ Target
correlation_values = X.corrwith(y)

# نستخدم القيمة المطلقة لأن -0.8 و +0.8 كلاهما علاقة قوية
absolute_correlation = correlation_values.abs()



# 4. second methode: Mutual Information


In [ ]:

# حساب Mutual Information لكل Feature مع Target
mutual_information_values = mutual_info_classif(
    X,
    y,
    random_state=42
)

mutual_information = pd.Series(
    mutual_information_values,
    index=feature_names
)

# third methode: Chi-Square

In [ ]:
# Chi-Square يتطلب أن تكون القيم غير سالبة.
# MinMaxScaler يحول القيم إلى المجال من 0 إلى 1.
scaler = MinMaxScaler()
X_non_negative = scaler.fit_transform(X)

chi_square_values, chi_square_pvalues = chi2(
    X_non_negative,
    y
)

chi_square = pd.Series(
    chi_square_values,
    index=feature_names
)

chi_square_pvalues = pd.Series(
    chi_square_pvalues,
    index=feature_names
)



# 4. Variance Threshold


In [ ]:
# حساب التباين لكل Feature
variance_values = X.var()

# يمكن تغيير قيمة الحد حسب المطلوب
variance_threshold_value = 0.0

# تطبيق Variance Threshold
variance_selector = VarianceThreshold(
    threshold=variance_threshold_value
)

variance_selector.fit(X)

# الميزات التي اجتازت الحد
selected_by_variance = X.columns[
    variance_selector.get_support()
].tolist()

# إنشاء عمود يوضح هل اجتازت الميزة الحد أم لا
variance_passed = pd.Series(
    X.columns.isin(selected_by_variance),
    index=feature_names
)

In [ ]:

results = pd.DataFrame({
    "Feature": feature_names,

    "Correlation": correlation_values.values,

    "Absolute_Correlation": absolute_correlation.values,

    "Mutual_Information": mutual_information.values,

    "Chi_Square": chi_square.values,

    "Chi_Square_P_Value": chi_square_pvalues.values,

    "Variance": variance_values.values,

    "Passed_Variance_Threshold": variance_passed.values
})

In [ ]:
# كلما كانت القيمة أكبر كان الترتيب أفضل
results["Correlation_Rank"] = (
    results["Absolute_Correlation"]
    .rank(ascending=False, method="min")
)

results["Mutual_Information_Rank"] = (
    results["Mutual_Information"]
    .rank(ascending=False, method="min")
)

results["Chi_Square_Rank"] = (
    results["Chi_Square"]
    .rank(ascending=False, method="min")
)

# في Variance: التباين الأكبر يأخذ ترتيبًا أعلى
results["Variance_Rank"] = (
    results["Variance"]
    .rank(ascending=False, method="min")
)


In [ ]:
# عدد الميزات التي تريد اختيارها
number_of_features = 10

# نستخدم Correlation وMutual Information وChi-Square
# لأن Variance لا تقيس العلاقة مع الـ Target،
# بل تقيس فقط مقدار تغير الميزة.
results["Average_Rank"] = results[
    [
        "Correlation_Rank",
        "Mutual_Information_Rank",
        "Chi_Square_Rank"
    ]
].mean(axis=1)

# نستبعد الميزات التي لم تتجاوز Variance Threshold
eligible_features = results[
    results["Passed_Variance_Threshold"] == True
].copy()

# ترتيب الميزات من الأفضل إلى الأضعف
eligible_features = eligible_features.sort_values(
    by="Average_Rank",
    ascending=True
)

# أخذ أفضل Features
best_features = eligible_features.head(
    number_of_features
)["Feature"].tolist()

In [ ]:
# =========================================================
# 10. طباعة النتائج
# =========================================================

print("\n" + "=" * 70)
print("جدول النتائج:")
print("=" * 70)

print(
    results[
        [
            "Feature",
            "Correlation",
            "Mutual_Information",
            "Chi_Square",
            "Variance",
            "Passed_Variance_Threshold"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print(f"أفضل {number_of_features} Features مقترحة:")
print("=" * 70)

for i, feature in enumerate(best_features, start=1):
    print(f"{i}. {feature}")


# # =========================================================
# # 11. حفظ النتائج في ملفات
# # =========================================================

# حفظ جدول النتائج بصيغة CSV
results.to_csv(
    "feature_selection_results.csv",
    index=False
)

# # حفظ جدول النتائج بصيغة Excel
# results.to_excel(
#     "feature_selection_results.xlsx",
#     index=False
# )

# حفظ الميزات المقترحة في ملف نصي
with open(
    "selected_features.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        f"أفضل {number_of_features} Features مقترحة:\n\n"
    )

    for i, feature in enumerate(best_features, start=1):
        file.write(f"{i}. {feature}\n")

# print("\nتم حفظ الملفات التالية:")
# print("1. feature_selection_results.csv")
# print("2. feature_selection_results.xlsx")
# print("3. selected_features.txt")



جدول النتائج:
                Feature  Correlation  Mutual_Information  Chi_Square      Variance  Passed_Variance_Threshold
            Radius_Mean     0.730029            0.362276   24.897293     12.418920                       True
           Texture_Mean     0.415185            0.096540    6.394071     18.498909                       True
         Perimeter_Mean     0.742636            0.402361   26.528902    590.440480                       True
              Area_Mean     0.708984            0.360023   29.328594 123843.554318                       True
        Smoothness_Mean     0.358560            0.079740    2.981900      0.000198                       True
       Compactness_Mean     0.596534            0.213439   20.353176      0.002789                       True
         Concavity_Mean     0.696360            0.375447   46.186395      0.006355                       True
    Concave_Points_Mean     0.776614            0.438806   52.405743      0.001506                       